# 01: Neural Network Basics

Build and train a simple neural network using PyTorch.

## Topics
- Neural network structure (layers, activations)
- Forward pass
- Loss functions and optimizers
- Training loop
- Visualizing training loss

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Generate Synthetic Classification Data

We create a dataset with 2 features and 3 classes. This lets us visualize the decision boundary later.

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=1000,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_classes=3,
    n_clusters_per_class=1,
    random_state=42,
    class_sep=1.5
)

print(f"Data shape: {X.shape}")
print(f"Labels: {np.unique(y)}")

# Visualize the data
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', alpha=0.6, edgecolors='w', s=50)
plt.colorbar(scatter, label='Class')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Synthetic Classification Data (3 classes)')
plt.show()

## 2. Prepare PyTorch DataLoaders

In [ ]:
# Convert to PyTorch tensors
X_tensor = torch.FloatTensor(X)
y_tensor = torch.LongTensor(y)

# Train/test split (80/20)
train_size = int(0.8 * len(X))
X_train, X_test = X_tensor[:train_size], X_tensor[train_size:]
y_train, y_test = y_tensor[:train_size], y_tensor[train_size:]

# Create DataLoaders
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## 3. Define the Neural Network

A simple feedforward network:
- Input: 2 features
- Hidden layer 1: 64 neurons, ReLU activation
- Hidden layer 2: 32 neurons, ReLU activation
- Output: 3 neurons (one per class)

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, 64),    # Input layer -> Hidden layer 1
            nn.ReLU(),            # Activation function
            nn.Linear(64, 32),   # Hidden layer 1 -> Hidden layer 2
            nn.ReLU(),            # Activation function
            nn.Linear(32, 3),    # Hidden layer 2 -> Output layer
        )

    def forward(self, x):
        return self.layers(x)

model = SimpleNet()
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Understand the Forward Pass

Before training, let's see what the model outputs with random weights.

In [ ]:
# Forward pass on a single sample
sample = X_train[0:1]  # Shape: [1, 2]
print(f"Input shape: {sample.shape}")
print(f"Input values: {sample}")

output = model(sample)
print(f"\nRaw output (logits): {output}")
print(f"Output shape: {output.shape}")

# Apply softmax to get probabilities
probs = torch.softmax(output, dim=1)
print(f"\nProbabilities: {probs}")
print(f"Sum of probabilities: {probs.sum().item():.4f}")
print(f"Predicted class: {output.argmax(dim=1).item()}")

## 5. Define Loss Function and Optimizer

- **CrossEntropyLoss**: Standard loss for multi-class classification (combines LogSoftmax + NLLLoss)
- **Adam optimizer**: Adaptive learning rate optimizer (recommended default)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam")
print(f"Learning rate: 0.001")

## 6. Training Loop

The standard training loop:
1. Forward pass — compute predictions
2. Compute loss
3. Backward pass — compute gradients
4. Update weights
5. Zero gradients

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_X, batch_y in loader:
        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        # Backward pass
        optimizer.zero_grad()  # Reset gradients
        loss.backward()        # Compute gradients
        optimizer.step()       # Update weights

        # Track metrics
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(batch_y).sum().item()
        total += batch_y.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_X, batch_y in loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(batch_y).sum().item()
            total += batch_y.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy

In [ ]:
# Train the model
num_epochs = 50
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    test_loss, test_acc = evaluate(model, test_loader, criterion)

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.1f}% | "
              f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.1f}%")

## 7. Visualize Training Progress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot loss
axes[0].plot(train_losses, label='Train Loss', linewidth=2)
axes[0].plot(test_losses, label='Test Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Test Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot accuracy
axes[1].plot(train_accuracies, label='Train Accuracy', linewidth=2)
axes[1].plot(test_accuracies, label='Test Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Test Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Visualize Decision Boundary

See how the network separates the classes in feature space.

In [ ]:
model.eval()

# Create a mesh grid
h = 0.02  # step size
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Predict on all points in the mesh
grid_points = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])
with torch.no_grad():
    Z = model(grid_points).argmax(dim=1).numpy()
Z = Z.reshape(xx.shape)

# Plot
plt.figure(figsize=(10, 8))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', edgecolors='w', s=50, alpha=0.8)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Neural Network Decision Boundary')
plt.show()

## Key Takeaways

1. **Network structure**: Layers of neurons with activation functions learn complex patterns
2. **Forward pass**: Data flows input → output through weighted connections
3. **Loss function**: Measures how wrong predictions are
4. **Backpropagation**: Computes gradients to update weights
5. **Training loop**: Repeatedly forward, loss, backward, update
6. **The network learns** non-linear decision boundaries that separate classes